# 05 · Charts & getting ready for Streamlit 📈

Numbers in a table are useful; a **picture** is instant. In this last notebook
you'll draw charts from your data, then see how the very same code becomes your
**Streamlit stock screener**. The chart you build here is the **drill-down** view —
the picture the user sees after picking one stock from the screener.

## What you'll learn
- Plotting a column with `.plot()` (uses **matplotlib**)
- Drawing a clean line chart of closing prices
- Computing a **moving average** and plotting it *with* the price
- How the notebook ideas become a **Streamlit screener** — a filterable table plus
  a drill-down chart
- Where to go next: the SPEC, the guides, NEXT-STEPS, and the `solution` branch

> 🌐 We download data again so this notebook stands alone. You do **not** need to
> install or run Streamlit here — that comes when you build `app.py`.

## 0. Get one stock to chart

The screener lists **many** stocks; when the user picks one, they see a chart of
just that stock — the **drill-down** view. So here we're back to the familiar
**one-ticker** download (flat columns), which means `data["Close"]` is a single
column we can plot directly.

In [ ]:
import yfinance as yf

data = yf.download("AAPL", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
data.tail()

## 1. The quickest chart — `.plot()`

Any pandas Series or DataFrame has a `.plot()` method. Calling it on the `Close`
column draws the closing price over time — the date index becomes the x-axis
automatically.

In [ ]:
import matplotlib.pyplot as plt

data["Close"].plot(title="AAPL — closing price")
plt.ylabel("Price ($)")
plt.show()

That's it — one line of real charting code. If you don't see a chart, make sure
you ran the cell (Shift + Enter); the plot appears just beneath it.

### 🏋️ Your turn

Plot the **`Volume`** column instead, with the title `"AAPL — daily volume"`.

In [ ]:
# TODO: call .plot(...) on data["Volume"] with a title, then plt.show().

**One way to do it:**

```python
data["Volume"].plot(title="AAPL — daily volume")
plt.ylabel("Shares traded")
plt.show()
```

## 2. A moving average — smoothing the noise

Daily prices jump around. A **moving average** smooths them by averaging the last
N days at each point, so the trend is easier to see. pandas does it in one call:
`.rolling(window).mean()`.

Here's a 20-day moving average (roughly one trading month).

In [ ]:
data["MA20"] = data["Close"].rolling(20).mean()
data[["Close", "MA20"]].tail()      # first 19 MA20 values are blank (NaN) — not enough days yet

The first 19 rows of `MA20` are empty (`NaN`) because a 20-day average needs 20
days of history before it can start. That's expected.

## 3. Price and moving average on one chart

Because `Close` and `MA20` are both columns now, plotting **both together** is as
easy as selecting them and calling `.plot()`. pandas draws one line per column and
adds a legend.

In [ ]:
data[["Close", "MA20"]].plot(title="AAPL — price vs 20-day average")
plt.ylabel("Price ($)")
plt.show()

### 🏋️ Your turn

Add a **50-day** moving average as a new column `MA50`, then plot `Close`, `MA20`
and `MA50` on the same chart. (A longer window = a smoother, slower line.)

In [ ]:
# TODO: create data["MA50"] with rolling(50), then plot the three columns together.

**One way to do it:**

```python
data["MA50"] = data["Close"].rolling(50).mean()
data[["Close", "MA20", "MA50"]].plot(title="AAPL — price with two moving averages")
plt.ylabel("Price ($)")
plt.show()
```

## 4. From these notebooks to a screener app 🚀

Here's the exciting part: **you've already written every hard bit.** Streamlit just
wraps them so they run in a web page and react to clicks. The screener has **two
views**:

1. **The universe table** — every stock, one row each, that you can filter, sort
   and search (notebooks 03 + 04).
2. **The drill-down chart** — pick one stock and see its price line (this notebook).

| In these notebooks | In the Streamlit screener (`app.py`) |
| ------------------ | ------------------------------------ |
| build the `summary` / `universe` table (nb 03) | `st.dataframe(universe)` — the whole screener |
| `universe[universe["Change %"] > 0]` | a sidebar `st.slider("Min change %", ...)` |
| `universe[universe["Name"].str.contains(text)]` | a sidebar `st.text_input("Search name")` |
| `universe.sort_values("Change %", ...)` | click a column header to sort the table |
| choosing one ticker | `st.selectbox("Drill into", universe.index)` |
| `data["Close"].plot()` (this notebook) | `st.line_chart(data["Close"])` for that stock |

**Widgets** are the magic word. `st.slider(...)`, `st.text_input(...)` and
`st.selectbox(...)` draw a control on the page and hand back whatever the user
chose — as a normal Python variable. So your filtering and charting code doesn't
change at all; it just receives *user-chosen* values instead of hard-coded ones.

A tiny taste of what the screener `app.py` looks like (you don't run this here — it
needs `streamlit run`). Notice the **two** downloads: the multi-ticker one builds
the universe table (multi-level columns), and the single-ticker drill-down keeps
`multi_level_index=False` so `data["Close"]` stays one column:

```python
import streamlit as st
import yfinance as yf
import pandas as pd

st.title("🔎 Stock Screener")

# 1. The universe — one row per company (notebook 03)
tickers = {"AAPL": "Apple", "MSFT": "Microsoft", "TSLA": "Tesla",
           "NVDA": "Nvidia", "KO": "Coca-Cola"}
raw = yf.download(list(tickers), period="6mo", auto_adjust=True, progress=False)
close = raw["Close"]
universe = pd.DataFrame({
    "Name": pd.Series(tickers),
    "Price": close.iloc[-1],
    "Change %": (close.iloc[-1] / close.iloc[0] - 1) * 100,
    "Avg Volume": raw["Volume"].mean(),
}).round(2)

# 2. Sidebar filters — your masks from notebook 04
min_change = st.sidebar.slider("Min change %", -50, 50, 0)
search = st.sidebar.text_input("Search name", "")
view = universe[universe["Change %"] >= min_change]
if search:
    view = view[view["Name"].str.contains(search)]
st.dataframe(view)                                  # the screener table

# 3. Drill into one stock — your chart from this notebook
pick = st.selectbox("Show chart for", view.index)
data = yf.download(pick, period="6mo", auto_adjust=True,
                   multi_level_index=False, progress=False)
st.line_chart(data["Close"])                        # the drill-down chart
```

Run the real app with `uv run streamlit run app.py` from the project folder.

## 🎓 You made it — now go build it

You've learned the whole toolkit: Python basics, pandas tables, downloading data,
filtering, and charting. That's genuinely everything the **screener** needs. The
rest is assembling these pieces in `app.py`, one step at a time.

**Where to go next:**
- 📖 **[`../guides/05-building-the-dashboard.md`](../guides/05-building-the-dashboard.md)**
  — the friendly walkthrough for turning these ideas into `app.py`.
- 📋 **[`../SPEC.md`](../SPEC.md)** — the job spec: what the screener should do.
- 🧭 **[`../NEXT-STEPS.md`](../NEXT-STEPS.md)** — where to take it once the basics work.
- 💡 **The `solution` branch** — a complete working screener to peek at *after*
  you've tried: `git switch solution`. There's no single right answer; compare and
  learn.

Now open `app.py` and start building. You've got this. 🎉